# 03 Inspection Level Preprocessing

This notebook restructures the data so each row represents a single inspection and prepares supporting preprocessing outputs used later in feature engineering.

In [2]:
# Author: Jessica
# Loading cleaned data so this notebook runs independently

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_filtered_2022_2025 = pd.read_csv("shared_data/cleaned_inspections.csv", parse_dates=['INSPECTION DATE'])
print(df_filtered_2022_2025.shape)
df_filtered_2022_2025.head()

(267210, 27)


,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,...,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
0,40837042,MALECON RESTAURANT,Bronx,55925594,BROADWAY,NaN,7184325155,Latin American,2025-04-16,Violations were cited in the following area(s).,...,Cycle Inspection / Re-inspection,0.000000,0.000000,NaN,NaN,NaN,NaN,2.000000e+00,NaN,NaN
1,50129114,PRIMOS BAKERY CAFE,Queens,Q3-015,JUNCTION BLVD,11103.0,7182713183,Bakery Products/Desserts,2023-08-22,Violations were cited in the following area(s).,...,Pre-permit (Operational) / Re-inspection,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,50151200,FLIP,0,1000,3rd Ave,NaN,9172241151,American,2025-01-15,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,50171281,LOVE MI YOGURT,Manhattan,172 1/2,CANAL STREET,10013.0,9178183268,Frozen Desserts,2025-06-23,Violations were cited in the following area(s).,...,Pre-permit (Operational) / Initial Inspection,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,50122846,IPIZZANY,Manhattan,621,2 AVENUE,10016.0,6465967071,Pizza,2023-10-12,Establishment re-opened by DOHMH.,...,Pre-permit (Operational) / Reopening Inspection,40.744677,-73.975961,106.0,2.0,7000.0,1020129.0,1.009140e+09,MN20,POINT (-73.97596120472 40.744677255478)


## Scope

This notebook contains preprocessing work that prepares the filtered inspection data for feature engineering. It includes chain versus independent business cleanup, categorical encoding experiments, text preprocessing helpers, and the aggregation steps that convert violation-level data into inspection-level data.

Fixing chain 

In [3]:
# Clean DBA names to reduce formatting duplicates
#author: Jess
df_filtered_2022_2025['DBA_clean'] = (
    df_filtered_2022_2025['DBA']
    .str.upper()
    .str.strip()
)
 
# Count unique restaurant locations (CAMIS) per DBA
dba_unique_locations = (
    df_filtered_2022_2025
    .groupby('DBA_clean')['CAMIS']
    .nunique()
)

# Identify chains as DBAs with more than one unique location
chain_names = set(dba_unique_locations[dba_unique_locations > 1].index)

# Create binary feature: 1 = chain, 0 = independent
df_filtered_2022_2025['is_chain'] = (
    df_filtered_2022_2025['DBA_clean']
    .isin(chain_names)
    .astype(int)
)

# Create numeric feature for chain size
df_filtered_2022_2025['chain_size'] = (
    df_filtered_2022_2025['DBA_clean']
    .map(dba_unique_locations)
)

# Display summary
print("Chain vs Independent distribution:")
print(df_filtered_2022_2025['is_chain'].value_counts())

print("\nTop 10 chains by number of locations:")
print(dba_unique_locations.sort_values(ascending=False).head(10))




Chain vs Independent distribution:
is_chain
0    195195
1     72015
Name: count, dtype: int64

Top 10 chains by number of locations:
DBA_clean
DUNKIN                    348
STARBUCKS                 197
SUBWAY                    168
MCDONALD'S                165
POPEYES                   122
DUNKIN'                    96
CHIPOTLE MEXICAN GRILL     72
DOMINO'S                   71
TACO BELL                  67
BURGER KING                65
Name: CAMIS, dtype: int64


In [4]:
!pip install scikit-learn

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = ["INSPECTION TYPE","BORO"]
# newer sklearn uses sparse_output instead of sparse

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preproc = ColumnTransformer(
    transformers=[
        ("cat", ohe, cat_cols),
    ],
    remainder="passthrough"
)

# fit_transform on a subset (example)
X = df_filtered_2022_2025[cat_cols]
X_enc = preproc.fit_transform(X)
print("Transformed shape:", X_enc.shape)
print("Feature names:", preproc.get_feature_names_out())


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Transformed shape: (267210, 39)
Feature names: ['cat__INSPECTION TYPE_Administrative Miscellaneous / Compliance Inspection'
 'cat__INSPECTION TYPE_Administrative Miscellaneous / Initial Inspection'
 'cat__INSPECTION TYPE_Administrative Miscellaneous / Re-inspection'
 'cat__INSPECTION TYPE_Administrative Miscellaneous / Reopening Inspection'
 'cat__INSPECTION TYPE_Administrative Miscellaneous / Second Compliance Inspection'
 'cat__INSPECTION TYPE_Calorie Posting / Compliance Inspection'
 'cat__INSPECTION TYPE_Calorie Posting / Initial Inspection'
 'cat__INSPECTION TYPE_Calorie Posting / Re-inspection'
 'cat__INSPECTION TYPE_Cycle Inspection / Compliance Inspection'
 'cat__INSPECTION TYPE_Cycle Inspection / Initial Inspection'
 'cat__INSPECTION TYPE_Cycle Inspection / Re-inspection'
 'cat__INSPECTION TYPE_Cycle Inspection / Reopening Inspection'
 'cat__INSPECTION TYPE_Cycle Insp

In [5]:
# Frequency encoding for violation codes
# calculate the frequency of each violation code in the filtered dataset

vc_counts = df_filtered_2022_2025['VIOLATION CODE'].value_counts()
# map counts back to the dataframe as a new column

df_filtered_2022_2025['violation_freq'] = df_filtered_2022_2025['VIOLATION CODE'].map(vc_counts)

# Optionally normalize to proportion

df_filtered_2022_2025['violation_freq_pct'] = (
    df_filtered_2022_2025['violation_freq'] / len(df_filtered_2022_2025)
)

print("Top 10 violation codes by frequency:")
print(vc_counts.head(10))
print(df_filtered_2022_2025[['VIOLATION CODE', 'violation_freq', 'violation_freq_pct']].head())

Top 10 violation codes by frequency:
VIOLATION CODE
10F    37505
08A    24768
06D    17129
10B    16927
02G    16866
06C    16469
02B    14206
04L    14102
04N    11007
04A     7293
Name: count, dtype: int64
  VIOLATION CODE  violation_freq  violation_freq_pct
0            09E          1958.0            0.007328
1            09E          1958.0            0.007328
2            02B         14206.0            0.053164
3            04L         14102.0            0.052775
4            NaN             NaN                 NaN


In [6]:
# Target encoding for cuisine description
# use a numeric target (e.g. inspection score) or convert grade to numeric

# install helper package if not available
try:
    import category_encoders as ce
except ImportError:
    import sys
    !{sys.executable} -m pip install category_encoders
    import category_encoders as ce

# choose the target column for encoding
target_col = "SCORE"  # or use "GRADE" after mapping grades to numbers

# drop rows with missing target (or alternatively fill them)
mask = df_filtered_2022_2025[target_col].notna()
if mask.sum() < len(df_filtered_2022_2025):
    print(f"Dropping {len(df_filtered_2022_2025) - mask.sum()} rows with null {target_col}")
df_te = df_filtered_2022_2025.loc[mask].copy()

# build and apply the encoder
te = ce.TargetEncoder(cols=["CUISINE DESCRIPTION"], smoothing=0.3)

df_te["cuisine_te"] = te.fit_transform(
    df_te[["CUISINE DESCRIPTION"]],
    df_te[target_col],
)

# merge back if desired (here just show the result)
print(df_te[["CUISINE DESCRIPTION", "cuisine_te"]].head())


Dropping 11922 rows with null SCORE
        CUISINE DESCRIPTION  cuisine_te
0            Latin American   27.954848
1  Bakery Products/Desserts   24.824567
2                  American   22.630114
3           Frozen Desserts   20.775684
4                     Pizza   24.341460


In [7]:
# text preprocessing helper
import re
from sklearn.feature_extraction.text import TfidfVectorizer

def clean_text(s):
    if pd.isna(s): return ""
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return " ".join(s.split())

# apply to violation descriptions
df_filtered_2022_2025["viol_clean"] = df_filtered_2022_2025["VIOLATION DESCRIPTION"].apply(clean_text)
tfidf = TfidfVectorizer(max_features=500)
viol_tfidf = tfidf.fit_transform(df_filtered_2022_2025["viol_clean"])

# business name patterns
df_filtered_2022_2025["name_len"] = df_filtered_2022_2025["DBA_clean"].str.len()
df_filtered_2022_2025["name_has_digit"] = df_filtered_2022_2025["DBA_clean"].str.contains(r"\d").astype(int)

# address features
addr = df_filtered_2022_2025["BUILDING"].fillna("") + " " + df_filtered_2022_2025["STREET"].fillna("")
df_filtered_2022_2025["street_num"] = addr.str.extract(r"^(\d+)")
df_filtered_2022_2025["street_name"] = addr.str.extract(r"^\d+\s+(.*)")

# …and you could join with an external geo‑lookup table for neighbourhoods

In [8]:
#Start of Feature Engineering
#Starting Author: Joel Barnes
#Data Frame: df_filtered_2022_2025
#Part 1: Violation Category Aggregations
# Extract violation category from code prefix
df_filtered_2022_2025['VIOLATION CATEGORY'] = df_filtered_2022_2025['VIOLATION CODE'].str[:2]


# Map to human-readable categories
category_map = {
    '02': 'food temperature',
    '03': 'food temperature',
    '04': 'food handling',
    '05': 'food handling',
    '06': 'personal hygiene',
    '08': 'facility conditions',
    '09': 'facility conditions',
    '10': 'pest control',
    '15': 'chemical toxin',
    '20': 'trans fat',
    '22': 'calorie posting'
}


df_filtered_2022_2025['VIOLATION CATEGORY LABEL'] = df_filtered_2022_2025['VIOLATION CATEGORY'].map(category_map).fillna('other')


In [9]:
violation_counts = df_filtered_2022_2025.groupby(['CAMIS', 'INSPECTION DATE', 'VIOLATION CATEGORY LABEL']).size().unstack(fill_value=0)
violation_counts.columns = [f'viol_{col}_count' for col in violation_counts.columns]
violation_counts = violation_counts.reset_index()


In [10]:
#Part 2: Critical Levels
# Binary flag for critical violations
df_filtered_2022_2025['IS CRITICAL'] = df_filtered_2022_2025['CRITICAL FLAG'].str.upper().str.strip() == 'CRITICAL'


# Per-inspection critical aggregations
critical_agg = df_filtered_2022_2025.groupby(['CAMIS', 'INSPECTION DATE']).agg(
    total_violations       = ('VIOLATION CODE', 'count'),
    critical_violations    = ('IS CRITICAL', 'sum'),
    noncritical_violations = ('IS CRITICAL', lambda x: (~x).sum()),
    unique_categories      = ('VIOLATION CATEGORY LABEL', 'nunique')
).reset_index()


# Ratio feature — proportion of violations that are critical
critical_agg['critical_ratio'] = (
    critical_agg['critical_violations'] / critical_agg['total_violations'].replace(0, np.nan)
).fillna(0)


# Flag: any critical violation present
critical_agg['has_critical'] = (critical_agg['critical_violations'] > 0).astype(int)


In [11]:
#Part 3: Historical Aggregations Per Restaurant
df_filtered_2022_2025_sorted = df_filtered_2022_2025.sort_values(['CAMIS', 'INSPECTION DATE'])


# Get one row per inspection with score
inspection_level = df_filtered_2022_2025_sorted.groupby(['CAMIS', 'INSPECTION DATE']).agg(
    score = ('SCORE', 'first'),
    grade = ('GRADE', 'first')
).reset_index()


inspection_level = inspection_level.merge(critical_agg, on=['CAMIS', 'INSPECTION DATE'])
inspection_level = inspection_level.merge(violation_counts, on=['CAMIS', 'INSPECTION DATE'])
inspection_level = inspection_level.sort_values(['CAMIS', 'INSPECTION DATE'])


# Expanding window historical features (no leakage)
inspection_level['historical_avg_score'] = (
    inspection_level.groupby('CAMIS')['score']
    .apply(lambda x: x.shift(1).expanding().mean())
    .reset_index(level=0, drop=True)
)


inspection_level['historical_avg_critical'] = (
    inspection_level.groupby('CAMIS')['critical_violations']
    .apply(lambda x: x.shift(1).expanding().mean())
    .reset_index(level=0, drop=True)
)


inspection_level['prior_inspection_count'] = (
    inspection_level.groupby('CAMIS').cumcount()  # 0-indexed, so this = number of prior inspections
)


## Saved Output

This notebook saves the inspection-level dataset used in feature engineering.

In [12]:
# Author: Jessica
# Saving inspection-level data so later notebooks can run independently

inspection_level_path = "shared_data/inspection_level.csv"
inspection_level.to_csv(inspection_level_path, index=False)
print(f"Saved inspection-level dataset to {inspection_level_path}")

Saved inspection-level dataset to shared_data/inspection_level.csv
